# FAMEWS Fairness Audit (Quick Start)

This notebook shows a minimal end-to-end example for:

1. Loading `HiRIDDataset`
2. Running `FAMEWS_fairness_audit` on a patient
3. Building a `SampleDataset` with `set_task(...)`
4. Printing a quick subgroup summary (sex and age group)

Use `dev=True` for a fast smoke test, then switch to `dev=False` for full runs.

In [ ]:
from collections import Counter
from pathlib import Path
import sys

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "pyhealth").exists():
            return path
    raise FileNotFoundError("Could not locate the PyHealth repository root from the current working directory.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
for name in list(sys.modules):
    if name == "pyhealth" or name.startswith("pyhealth."):
        del sys.modules[name]
if str(REPO_ROOT) in sys.path:
    sys.path.remove(str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT))

from pyhealth.datasets.hirid import HiRIDDataset
from pyhealth.tasks.HiRID_fairness_audit import FAMEWS_fairness_audit

HIRID_ROOT = REPO_ROOT / "test-resources" / "core" / "hiriddemo"

assert HIRID_ROOT.exists(), (
    f"Expected HiRID root at {HIRID_ROOT}, but it was not found."
)

print(f"Using HiRID root: {HIRID_ROOT}")

: 

In [ ]:
# Quick setup: use imputed stage + dev mode for fast iteration.
dataset = HiRIDDataset(
    root=str(HIRID_ROOT),
    stage="imputed",
    dev=True,
)

task = FAMEWS_fairness_audit(stage_table="imputed_stage")

# Inspect one raw task sample from a single patient.
first_pid = dataset.unique_patient_ids[0]
first_patient = dataset.get_patient(first_pid)
raw_samples = task(first_patient)

print(f"First patient id: {first_pid}")
print(f"Raw samples generated for first patient: {len(raw_samples)}")

if raw_samples:
    raw0 = raw_samples[0]
    print("Raw sample keys:", sorted(raw0.keys()))
    print("Demographics:", {
        "sex": raw0.get("sex"),
        "age": raw0.get("age"),
        "age_group": raw0.get("age_group"),
        "discharge_status": raw0.get("discharge_status"),
    })

    ts, values = raw0["signals"]
    print("Timeseries length:", len(ts))
    print("Values shape:", values.shape)

    display(pd.DataFrame(values[:5], columns=raw0["feature_columns"]))

In [ ]:
# Build a PyHealth SampleDataset (this applies processors and caches outputs).
sample_dataset = dataset.set_task(task, num_workers=1)

print("Dataset stats:")
dataset.stats()
print(f"\nNumber of processed ML samples: {len(sample_dataset)}")

sample0 = sample_dataset[0]
print("SampleDataset keys:", sorted(sample0.keys()))

In [ ]:
# Quick fairness-oriented summary over a small cohort slice.
# We intentionally use raw task outputs here to keep metadata fields explicit.
max_patients = 200
sex_counter = Counter()
age_group_counter = Counter()
valid_patients = 0

for pid in dataset.unique_patient_ids[:max_patients]:
    p = dataset.get_patient(pid)
    samples = task(p)
    if not samples:
        continue
    valid_patients += 1
    s = samples[0]
    sex_counter[str(s.get("sex"))] += 1
    age_group_counter[str(s.get("age_group"))] += 1

print(f"Patients scanned: {max_patients}")
print(f"Patients with at least one task sample: {valid_patients}")

summary_df = pd.DataFrame({
    "sex": dict(sex_counter),
    "age_group": dict(age_group_counter),
})

display(summary_df.fillna(0).astype(int))